<a href="https://colab.research.google.com/github/leovarconrenno/Estacao-de-Reabastecimento-de-Hidrogenio---SCADA-Core/blob/main/etapa-02-grafos/16%20-%20Problemas%20Eulerianos%20e%20Inspecao%20de%20Infraestrutura.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 16 - Notebook: Circuitos Eulerianos e Inspeção Autônoma de Tubulações de H₂

Neste notebook modelamos a malha física de inspeção periódica da Estação de Reabastecimento de Hidrogênio e implementamos o **Algoritmo de Hierholzer** para gerar a rota ótima do robô *crawler* / PIG ultrassônico de detecção de fragilização por H₂, garantindo cobertura completa de 100% das linhas sem repetição de trechos.

![Grafo da Estação de Hidrogênio](grafo_tubulacoes.png)

In [1]:
import sys
sys.stdout.reconfigure(encoding='utf-8')

def formatar_tabela(dados):
    # Formata lista de dicionarios em tabela ASCII pura
    if not dados:
        return 'Tabela Vazia'
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ''))))
    header = ' | '.join(f'{c:<{larguras[c]}}' for c in colunas)
    divisor = '-+-'.join('-' * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(' | '.join(f'{str(row.get(c, "")):<{larguras[c]}}' for c in colunas))
    return '\n'.join(linhas)


from collections import defaultdict
from typing import List, Tuple


class GrafoInspecaoEuleriano:
    def __init__(self):
        self.adj = defaultdict(list)
        self.dutos_info = []

    def adicionar_duto(self, u: str, v: str, id_duto: str, tag_valvula: str = '', comp_m: float = 0.0):
        self.adj[u].append((v, id_duto))
        self.adj[v].append((u, id_duto))
        self.dutos_info.append({
            'Duto': id_duto, 'Trecho': f'{u} <-> {v}',
            'Componente_ISA': tag_valvula, 'Comprimento_m': comp_m
        })

    def verificar_euleriano(self) -> Tuple[bool, List[str]]:
        impares = [v for v, viz in self.adj.items() if len(viz) % 2 != 0]
        return len(impares) == 0, impares

    def calcular_circuito_hierholzer(self, inicio: str) -> List[str]:
        adj_copia = {u: list(viz) for u, viz in self.adj.items()}
        pilha = [inicio]
        circuito = []
        while pilha:
            u = pilha[-1]
            if adj_copia[u]:
                v, id_e = adj_copia[u].pop()
                adj_copia[v].remove((u, id_e))
                pilha.append(v)
            else:
                circuito.append(pilha.pop())
        circuito.reverse()
        return circuito


# Malha fisica de inspecao: 8 vertices e 10 dutos (todos de grau par: 2 ou 4)
g_insp = GrafoInspecaoEuleriano()
g_insp.adicionar_duto('BASE-100', 'SUP-100', 'd0_acesso', 'Trilho Acesso Base', 12.0)
g_insp.adicionar_duto('SUP-100', 'TK-101', 'd1_xv101', 'XV-101 (Alimentacao LP)', 10.0)
g_insp.adicionar_duto('SUP-100', 'TK-102', 'd2_xv102', 'XV-102 (Alimentacao MP)', 12.0)
g_insp.adicionar_duto('SUP-100', 'TK-103', 'd3_xv103', 'XV-103 (Alimentacao HP)', 15.0)
g_insp.adicionar_duto('TK-101', 'XV-104', 'd4_linha_lp', 'Linha LP (350 bar)', 8.0)
g_insp.adicionar_duto('TK-102', 'XV-104', 'd5_linha_mp', 'Linha MP (700 bar)', 6.0)
g_insp.adicionar_duto('TK-103', 'XV-104', 'd6_linha_hp', 'Linha HP (1000 bar)', 8.0)
g_insp.adicionar_duto('XV-104', 'HEX-201', 'd7_xv201', 'XV-201 (Entrada Chiller)', 14.0)
g_insp.adicionar_duto('HEX-201', 'DISP-300', 'd8_xv301', 'XV-301 (Linha Dispensador)', 16.0)
g_insp.adicionar_duto('DISP-300', 'BASE-100', 'd9_bv301', 'BV-301 (Retorno Base)', 14.0)

# Verificacao euleriana
eul, imp = g_insp.verificar_euleriano()
print(f'Grafo e Euleriano: {eul} | Vertices Impares: {imp}')

# Calculo do circuito de inspecao
rota_robo = g_insp.calcular_circuito_hierholzer('BASE-100')
print('Circuito Euleriano de Inspecao:', ' -> '.join(rota_robo))
print(f'Total de Trechos Inspecionados: {len(rota_robo) - 1} tubulacoes')

# Relatorio dos dutos inspecionados
print()
print('--- Detalhamento das Tubulacoes na Rota de Inspecao ---')
print(formatar_tabela(g_insp.dutos_info))

assert eul is True
assert len(imp) == 0
assert len(rota_robo) - 1 == 10


Grafo e Euleriano: True | Vertices Impares: []
Circuito Euleriano de Inspecao: BASE-100 -> DISP-300 -> HEX-201 -> XV-104 -> TK-103 -> SUP-100 -> TK-102 -> XV-104 -> TK-101 -> SUP-100 -> BASE-100
Total de Trechos Inspecionados: 10 tubulacoes

--- Detalhamento das Tubulacoes na Rota de Inspecao ---
Duto        | Trecho                | Componente_ISA             | Comprimento_m
------------+-----------------------+----------------------------+--------------
d0_acesso   | BASE-100 <-> SUP-100  | Trilho Acesso Base         | 12.0         
d1_xv101    | SUP-100 <-> TK-101    | XV-101 (Alimentacao LP)    | 10.0         
d2_xv102    | SUP-100 <-> TK-102    | XV-102 (Alimentacao MP)    | 12.0         
d3_xv103    | SUP-100 <-> TK-103    | XV-103 (Alimentacao HP)    | 15.0         
d4_linha_lp | TK-101 <-> XV-104     | Linha LP (350 bar)         | 8.0          
d5_linha_mp | TK-102 <-> XV-104     | Linha MP (700 bar)         | 6.0          
d6_linha_hp | TK-103 <-> XV-104     | Linha HP (1000 b